In [ ]:
import torch
import torch.nn as nn
from torch import Tensor
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader,SubsetRandomSampler,ConcatDataset
import torch.nn.functional as F

In [ ]:
from helper import plot_map, from_upper_triu, upper_triangular_to_vector_skip_diagonals

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

from model import SeqNN

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
# Load the entire model (architecture + weights)
model = torch.load("/home1/smaruj/pytorch_akita/model.pth")

In [ ]:
model = model.to(device)

In [ ]:
# Set the model to evaluation mode (important for inference)
model.eval()

In [ ]:
from torchinfo import summary

summary(model, input_size=(2, 4, 1048576), col_names=["output_size", "num_params"])

In [ ]:
X = torch.load("/scratch1/smaruj/ledidi_targets/constant_dot_background/X_0.pt", weights_only=True)

In [ ]:
X.shape

In [ ]:
y_bar = torch.load("/scratch1/smaruj/ledidi_targets/constant_dot_background/modified_y_dot_100_275_2.5p.pt", weights_only=True)

In [ ]:
# to ensure the local, forked ledidi is used
# not the one installed using pip

import sys
sys.path.insert(0, "/home1/smaruj/ledidi")  # Add the directory where "ledidi" is located
from ledidi import Ledidi

In [ ]:
X.shape, y_bar.shape

In [ ]:
wrapper = Ledidi(model, 
                 input_loss=torch.nn.L1Loss(reduction='sum'), 
                 output_loss=torch.nn.L1Loss(reduction='sum'),   # output_loss=torch.nn.MSELoss(),
                 batch_size=1,
                 max_iter=3000,
                 early_stopping_iter=2000,
                 return_history=True,
                 verbose=True,
                 bin_size=2048,
                 input_mask_slices_0=[100],
                 input_mask_slices_1=[275],
                 cropping_applied=32,
                 output_mask_path="./fragment_indices.pt",
                 use_semifreddo=False
                 ).cuda()

In [ ]:
# slice_torch = X[:,:,start_index : end_index]

In [ ]:
# X_l_flank = X[:,:,start_index - 2048*2 : start_index]
# X_r_flank = X[:,:,end_index : end_index + 2048*2]

In [ ]:
# x_bar, history = wrapper.fit_transform(X, modified_vector_tensor)
# if use_semifreddo=True
# x_bar = wrapper.fit_transform(slice_torch, y_bar, X_l_flank=X_l_flank, X_r_flank=X_r_flank)
# x_bar, history = wrapper.fit_transform(slice_torch, y_bar, X_l_flank=X_l_flank, X_r_flank=X_r_flank)

In [ ]:
# if use_semifreddo=False
# x_bar = wrapper.fit_transform(X, y_bar)
x_bar, history = wrapper.fit_transform(X, y_bar)

In [ ]:
bin_size = 2048
cropping_applied = 32
padding_bins = 2

slice_0_bins = [100]
slice_0_start = (min(slice_0_bins) + cropping_applied - padding_bins) * bin_size
slice_0_end = (max(slice_0_bins) + 1 + cropping_applied + padding_bins) * bin_size

slice_1_bins = [275]
slice_1_start = (min(slice_1_bins) + cropping_applied - padding_bins) * bin_size
slice_1_end = (max(slice_1_bins) + 1 + cropping_applied + padding_bins) * bin_size

In [ ]:
slice1 = x_bar[:, :, slice_0_start:slice_0_end]
slice2 = x_bar[:, :, slice_1_start:slice_1_end]

In [ ]:
slice_0_start,slice_0_end

In [ ]:
from semifreddo_full_model import Semifreddo

In [ ]:
semifreddo_model = Semifreddo(model=model,
                                          slice_0_padded_seq=slice1, 
                                          edited_indices_slice_0=[100],
                                          saved_temp_output_path="/scratch1/smaruj/ledidi_targets/constant_dot_background/tower_out.pt",
                                          slice_1_padded_seq=slice2,
                                          edited_indices_slice_1=[275],
                                          batch_size=1)
y_hat = semifreddo_model.forward()

In [ ]:
y_hat.shape

## Input and Output Loss Plots

In [ ]:
%matplotlib inline
import numpy
import matplotlib.pyplot as plt
import seaborn; seaborn.set_style('whitegrid')

plt.figure(figsize=(5, 3))
plt.plot(history['input_loss'], c='0.7', label="Input Loss")
plt.legend(fontsize=10)
plt.xlabel("Iterations")
plt.ylabel("Loss")

seaborn.despine(left=True, bottom=True)
plt.show()

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(history['output_loss'], c='0.3', label="Output Loss")
plt.legend(fontsize=10)
plt.xlabel("Iterations")
plt.ylabel("Loss")

seaborn.despine(left=True, bottom=True)
plt.show()

In [ ]:
# from ledidi.plot import plot_edits

# plot_edits(history)
# seaborn.despine(bottom=True, left=True)
# plt.show()

## Plotting edits

In [ ]:
X_new = X.clone()
X_new = X_new.repeat(10, 1, 1)

In [ ]:
X_new[:,:,start_index:end_index] = x_bar

In [ ]:
X_new = x_bar

In [ ]:
unique_set = set()

smallest_edited_location = 1048576
biggest_edited_location = 0

for i in range(1):

    print("sequence:", i)
    
    difference_vector = (X[0, :, :] == X_new[i, :, :]).all(dim=0).cpu().numpy()

    e=0
    edits_loc = []

    for loc_index in range(len(difference_vector)):
        if difference_vector[loc_index] == False:
            e += 1
            edits_loc.append(loc_index)
            if loc_index < smallest_edited_location:
                smallest_edited_location = loc_index
            if loc_index > biggest_edited_location:
                biggest_edited_location = loc_index
    
    edits_tuple = tuple(edits_loc)

    if edits_tuple not in unique_set:
        unique_set.add(edits_tuple)
    
    # print("number of edits:", e)
    # print("edits locations:", edits_loc)

In [ ]:
import numpy as np

In [ ]:
edits_array = np.array(edits_tuple) 
bins_array = edits_array // 2048

In [ ]:
bin_counts = np.bincount(bins_array)

In [ ]:
filtered_bins = np.arange(32, 480)
filtered_counts = bin_counts[filtered_bins]

In [ ]:
renumbered_bins = np.arange(0, 448)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(renumbered_bins, filtered_counts, color='blue', alpha=0.5)
plt.xlabel("Bin Index (Renumbered 0-448)")
plt.ylabel("Count")
plt.title("Bin Counts (Ignoring First & Last 32 Bins)")
plt.show()

In [ ]:
len(unique_set)

In [ ]:
import seaborn as sns
import numpy as np

In [ ]:
X_numpy = X_new.detach().cpu().numpy()
X_chunk = X_numpy[:, :, smallest_edited_location:biggest_edited_location]  # Shape: (10, 4, 200)

# Convert one-hot encoding to integer labels (0=A, 1=T, 2=C, 3=G)
sequence_labels = np.argmax(X_chunk, axis=1)  # Shape: (10, 200)

# Define a color map for nucleotides
nucleotide_cmap = {
    0: "green",  # A
    1: "blue",     # C
    2: "yellow",    # G
    3: "red"    # T
}

# Create a custom colormap for visualization
cmap = [nucleotide_cmap[i] for i in range(4)]
sns.set(style="white")

# Plot heatmap of all sequences
plt.figure(figsize=(10, 3))
sns.heatmap(sequence_labels, cmap=cmap, cbar=False, xticklabels=False, yticklabels=False)
plt.xlabel("Position")
plt.ylabel("Sequence")
plt.title("One-Hot Encoded DNA Sequences")
plt.show()

## CTCF & CTCF-flanks PWM score

In [ ]:
# CTCF PWM
CTCF_PWM = "/home1/smaruj/IterativeMutagenesis/MA0139.1.meme"

# CTCF with strong flanks (15bp)
CTCF_flanks = "/home1/smaruj/IterativeMutagenesis/PWM_with_flanks.npy"

In [ ]:
def read_meme_pwm_as_numpy(filename):
    pwm_list = []  # List to store PWM rows
    
    with open(filename, 'r') as file:
        in_matrix_section = False
        
        for line in file:
            line = line.strip()
            
            # Check if we are reading the PWM matrix
            if line.startswith("letter-probability matrix"):
                in_matrix_section = True  # Start reading matrix data
                continue  # Skip this header line
            
            # If we are in the matrix section, process the rows
            if in_matrix_section and line:
                pwm_row = [float(value) for value in line.split()]  # Parse values
                pwm_list.append(pwm_row)  # Append to the PWM list
            
            # If we encounter a new MOTIF or the end of file, stop matrix reading
            if line.startswith("MOTIF") and in_matrix_section:
                break
    
    # Convert the list to a numpy array
    pwm_array = np.array(pwm_list)
    
    return pwm_array

In [ ]:
pwm_CTCF = read_meme_pwm_as_numpy(CTCF_PWM)
pwm_CTCFflanks = np.load(CTCF_flanks)

In [ ]:
def pwm_convolution(pwm, sequence):
    len_pwm, _ = pwm.shape
    len_seq, _ = sequence.shape
    conv_scores = []
    
    # Sliding window over the sequence matrix
    for i in range(len_seq - len_pwm + 1):
        # Extract submatrix (window) of the sequence of size len_pwm x 4
        window = sequence[i:i+len_pwm, :]
        
        # Compute the dot product (element-wise multiplication and sum) for the window
        score = np.sum(pwm * window)
        conv_scores.append(score)
    
    # Return the maximum convolution score
    return conv_scores


def reverse_complement_pwm(pwm):
    """Get the reverse complement of a given PWM."""
    # Reverse the rows (i.e., reverse the motif) and swap A<->T (0<->3) and C<->G (1<->2)
    rev_comp_pwm = pwm[::-1, :][:, [3, 2, 1, 0]]
    return rev_comp_pwm


In [ ]:
x_bar_np = x_bar[:, :, :].cpu().detach().numpy().transpose(0, 2, 1)

In [ ]:
308 * 2048

In [ ]:
# Compute PWM scores for the first sequence
CTCF_PWM_score = pwm_convolution(pwm_CTCF, x_bar_np[0, 270336:272384, :])  
rev_CTCF_PWM_score = pwm_convolution(reverse_complement_pwm(pwm_CTCF), x_bar_np[0, 270336:272384, :])  

# Create a figure
plt.figure(figsize=(10, 5))

# Plot forward PWM score with transparency
plt.plot(CTCF_PWM_score, label=">", alpha=0.6)  

# Plot reverse PWM score with transparency
plt.plot(rev_CTCF_PWM_score, label="<", alpha=0.6)  

plt.axhline(y=11.1, color='black', linestyle='--', linewidth=1, label="average strong CTCF")

# Labels and title
plt.xlabel("Position")
plt.ylabel("PWM Score")
plt.title("PWM Convolution Scores for Sequence 1, 1st slice")
plt.legend(loc="lower right")

# Show the plot
plt.show()


In [ ]:
# Compute PWM scores for the first sequence
CTCF_PWM_score = pwm_convolution(pwm_CTCF, x_bar_np[0, 628736:630784, :])  
rev_CTCF_PWM_score = pwm_convolution(reverse_complement_pwm(pwm_CTCF), x_bar_np[0, 628736:630784, :])  

# Create a figure
plt.figure(figsize=(10, 5))

# Plot forward PWM score with transparency
plt.plot(CTCF_PWM_score, label=">", alpha=0.6)  

# Plot reverse PWM score with transparency
plt.plot(rev_CTCF_PWM_score, label="<", alpha=0.6)  

plt.axhline(y=11.1, color='black', linestyle='--', linewidth=1, label="average strong CTCF")

# Labels and title
plt.xlabel("Position")
plt.ylabel("PWM Score")
plt.title("PWM Convolution Scores for Sequence 1, 2nd slice")
plt.legend(loc="lower right")

# Show the plot
plt.show()


In [ ]:
# Compute PWM scores for the first sequence
CTCF_PWM_score = pwm_convolution(pwm_CTCFflanks, x_bar_np[0, 270336:272384, :])  
rev_CTCF_PWM_score = pwm_convolution(reverse_complement_pwm(pwm_CTCFflanks), x_bar_np[0, 270336:272384, :])  

# Create a figure
plt.figure(figsize=(10, 5))

# Plot forward PWM score with transparency
plt.plot(CTCF_PWM_score, label=">", alpha=0.6)  

# Plot reverse PWM score with transparency
plt.plot(rev_CTCF_PWM_score, label="<", alpha=0.6)  

plt.axhline(y=21.5, color='black', linestyle='--', linewidth=1, label="average strong CTCF")

# Labels and title
plt.xlabel("Position")
plt.ylabel("PWM (flanks) Score")
plt.title("PWM Convolution Scores (CTCF+flanks) for Sequence 1")
plt.legend(loc="lower right")

# Show the plot
plt.show()

In [ ]:
# Compute PWM scores for the first sequence
CTCF_PWM_score = pwm_convolution(pwm_CTCFflanks, x_bar_np[0, 628736:630784, :])  
rev_CTCF_PWM_score = pwm_convolution(reverse_complement_pwm(pwm_CTCFflanks), x_bar_np[0, 628736:630784, :])  

# Create a figure
plt.figure(figsize=(10, 5))

# Plot forward PWM score with transparency
plt.plot(CTCF_PWM_score, label=">", alpha=0.6)  

# Plot reverse PWM score with transparency
plt.plot(rev_CTCF_PWM_score, label="<", alpha=0.6)  

plt.axhline(y=21.5, color='black', linestyle='--', linewidth=1, label="average strong CTCF")

# Labels and title
plt.xlabel("Position")
plt.ylabel("PWM (flanks) Score")
plt.title("PWM Convolution Scores (CTCF+flanks) for Sequence 1")
plt.legend(loc="lower right")

# Show the plot
plt.show()

## Predicted Maps

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(x_bar, training=False)

In [ ]:
target_to_plot = 0

In [ ]:
matrix_to_plot = from_upper_triu(pred[0, target_to_plot, :], matrix_len=448, num_diags=2)

In [ ]:
matrix_to_plot = from_upper_triu(y_hat[0, target_to_plot, :], matrix_len=448, num_diags=2)

In [ ]:
import numpy as np

In [ ]:
np.mean(matrix_to_plot[97:103,273:278])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
width=5
height=5
vmin = -2
vmax = 2
palette="RdBu_r"

fig, axes = plt.subplots(1, 1, figsize=(width, height))

sns.heatmap(
    matrix_to_plot,
    vmin=vmin,
    vmax=vmax,
    cbar=True,
    cmap=palette,
    square=True,
    xticklabels=False,
    yticklabels=False,
    ax=axes
)

# Highlight the 194th row
axes.hlines(y=100, xmin=0, xmax=448, colors='black', linewidth=1.5)

# Highlight the 254th column
axes.vlines(x=275, ymin=0, ymax=448, colors='black', linewidth=1.5)

plt.tight_layout()
plt.show()

In [ ]:
width=5
height=5
vmin = -2
vmax = 2
palette="RdBu_r"

fig, axes = plt.subplots(1, 1, figsize=(width, height))

sns.heatmap(
    matrix_to_plot,
    vmin=vmin,
    vmax=vmax,
    cbar=True,
    cmap=palette,
    square=True,
    xticklabels=False,
    yticklabels=False,
    ax=axes
)

# Highlight the 194th row
axes.hlines(y=100, xmin=0, xmax=448, colors='black', linewidth=1.5)

# Highlight the 254th column
axes.vlines(x=275, ymin=0, ymax=448, colors='black', linewidth=1.5)

plt.tight_layout()
plt.show()

In [ ]:
for i in range(1):
    print(i)
    plot_map(from_upper_triu(pred[i, target_to_plot, :], matrix_len=448, num_diags=2), vmin=-0.6, vmax=0.6, palette="RdBu_r", width=5, height=5)